In [ ]:
# ============================================================
#  RAG Document Q&A Assistant
#  Author  : Narjes
#  Stack   : LangChain · PyMuPDF · ChromaDB · phi3 (Ollama) · Gradio
#  Version : 2.0
# ============================================================

# ── Standard Library ────────────────────────────────────────
import os
import logging

# ── LangChain ───────────────────────────────────────────────
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate

# ── UI ──────────────────────────────────────────────────────
import gradio as gr


# ════════════════════════════════════════════════════════════
#  CONFIGURATION
# ════════════════════════════════════════════════════════════

PDF_PATH       = r"C:\Users\pc\Desktop\Ai career accelerator course\Notebooks\Main Notebooks\Submitted Projects\rag assistant\notes.pdf"
CHROMA_DIR     = "./chroma_db"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
LLM_MODEL      = "phi3"
CHUNK_SIZE     = 500
CHUNK_OVERLAP  = 100
TOP_K          = 4

PROMPT_TEMPLATE = """
You are a precise and reliable assistant.

Answer ONLY using the provided context.
- If the answer is not clearly in the context, say: "I don't know."
- Keep answers concise and factual.

Context:
{context}

Question:
{question}

Answer:
"""


# ════════════════════════════════════════════════════════════
#  LOGGING
# ════════════════════════════════════════════════════════════

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
log = logging.getLogger(__name__)


# ════════════════════════════════════════════════════════════
#  PIPELINE SETUP
# ════════════════════════════════════════════════════════════

def load_documents(path: str):
    """Load all pages from a PDF file."""
    loader = PyMuPDFLoader(path)
    docs = loader.load()
    log.info("Loaded %d pages from '%s'", len(docs), path)
    return docs


def split_documents(documents):
    """Split documents into overlapping chunks for retrieval."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )
    chunks = splitter.split_documents(documents)
    log.info("Created %d chunks (size=%d, overlap=%d)", len(chunks), CHUNK_SIZE, CHUNK_OVERLAP)
    return chunks


def build_vectorstore(chunks, embeddings):
    """Load an existing ChromaDB or create a new one from chunks."""
    if os.path.exists(CHROMA_DIR):
        log.info("Loading existing vector store from '%s'", CHROMA_DIR)
        return Chroma(persist_directory=CHROMA_DIR, embedding_function=embeddings)

    log.info("Building new vector store at '%s'", CHROMA_DIR)
    return Chroma.from_documents(chunks, embeddings, persist_directory=CHROMA_DIR)


def setup_pipeline():
    """
    Full pipeline initialisation.
    Returns (retriever, llm, prompt).
    """
    # Step 1 – Load
    documents = load_documents(PDF_PATH)

    # Step 2 – Chunk
    chunks = split_documents(documents)

    # Step 3 – Embed + Store
    embeddings   = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
    vectorstore  = build_vectorstore(chunks, embeddings)

    # Step 4 – LLM  (make sure `ollama serve` is running)
    llm = Ollama(model=LLM_MODEL)
    log.info("LLM loaded: %s", LLM_MODEL)

    # Step 5 – Retriever
    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": TOP_K},
    )
    log.info("Retriever ready (top_k=%d)", TOP_K)

    # Step 6 – Prompt
    prompt = PromptTemplate(
        input_variables=["context", "question"],
        template=PROMPT_TEMPLATE,
    )

    log.info("Pipeline initialised successfully.")
    return retriever, llm, prompt


# ════════════════════════════════════════════════════════════
#  INITIALISE (runs once at startup)
# ════════════════════════════════════════════════════════════

retriever, llm, prompt = setup_pipeline()


# ════════════════════════════════════════════════════════════
#  CORE Q&A FUNCTION
# ════════════════════════════════════════════════════════════

def ask_question(question: str) -> str:
    """
    Retrieve relevant chunks, build a prompt, and return the LLM answer
    together with source excerpts.
    """
    if not question.strip():
        return "⚠️  Please enter a question."

    # Retrieve
    docs    = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)

    # Generate
    prompt_text = prompt.format(context=context, question=question)
    answer      = llm.invoke(prompt_text)

    # Format sources
    source_lines = ["\n\n─── Sources ───────────────────────────────"]
    for i, doc in enumerate(docs, start=1):
        excerpt = doc.page_content[:200].strip().replace("\n", " ")
        source_lines.append(f"\n[{i}] {excerpt}…")

    return str(answer) + "\n".join(source_lines)


# ════════════════════════════════════════════════════════════
#  GRADIO UI
# ════════════════════════════════════════════════════════════

def build_ui() -> gr.Interface:
    return gr.Interface(
        fn=ask_question,
        inputs=gr.Textbox(
            label="Your Question",
            placeholder="e.g. What is the main topic of this document?",
            lines=2,
        ),
        outputs=gr.Textbox(
            label="Answer",
            lines=10,
        ),
        title="📄 Document Q&A Assistant",
        description=(
            "Ask anything about your PDF notes.\n"
            f"**Model:** {LLM_MODEL} · **Embeddings:** {EMBEDDING_MODEL} · **Vector DB:** ChromaDB"
        ),
        allow_flagging="never",
    )


# ════════════════════════════════════════════════════════════
#  EVALUATION HELPER  (run manually when needed)
# ════════════════════════════════════════════════════════════

def run_evaluation():
    """Quick smoke-test with a few sample questions."""
    test_questions = [
        "What is the main topic?",
        "Summarize the document.",
        "What are the key conclusions?",
    ]
    separator = "=" * 60
    for question in test_questions:
        print(f"\n{separator}")
        print(f"QUESTION : {question}")
        print(f"ANSWER   : {ask_question(question)}")
    print(separator)


# ════════════════════════════════════════════════════════════
#  ENTRY POINT
# ════════════════════════════════════════════════════════════

if __name__ == "__main__":
    log.info("Launching Gradio UI → http://127.0.0.1:7860")
    ui = build_ui()
    ui.launch(share=False, inline=True)

2026-04-08 18:35:41,290 [INFO] Loaded 1 pages from 'C:\Users\pc\Desktop\Ai career accelerator course\Notebooks\Main Notebooks\Submitted Projects\rag assistant\notes.pdf'
2026-04-08 18:35:41,561 [INFO] Created 5 chunks (size=500, overlap=100)
2026-04-08 18:35:41,639 [INFO] Use pytorch device_name: cpu
2026-04-08 18:35:41,639 [INFO] Load pretrained SentenceTransformer: all-MiniLM-L6-v2
2026-04-08 18:35:42,296 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-04-08 18:35:42,386 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
2026-04-08 18:35:42,581 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-04-08 18:35:42,659 [INFO] HTTP Req

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


2026-04-08 18:35:49,856 [INFO] HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-launched-telemetry "HTTP/1.1 200 OK"
